In [1]:
import os
import google.generativeai as genai
from PIL import Image
import base64
import io
import fitz  # PyMuPDF
import os
from dotenv import load_dotenv
import csv
import json
import shutil
import time
import gc
from pathlib import Path

# ---
# Your existing setup and functions (with improvements)
# ---
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

# Configure the Gemini API
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
genai.configure(api_key=api_key)



model = genai.GenerativeModel(
    'gemini-2.5-pro',  # Changed to 1.5-pro for better stability
)

def convert_pdf_to_images(pdf_path, output_folder, dpi=120):  # Reduced DPI for faster processing
    """Converts each page of a PDF to a PNG image."""
    os.makedirs(output_folder, exist_ok=True)
    image_paths = []
    try:
        pdf_document = fitz.open(pdf_path)
        for page_number in range(len(pdf_document)):
            page = pdf_document[page_number]
            # Reduced resolution for faster processing and smaller file sizes
            pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
            output_path = os.path.join(output_folder, f"page_{page_number + 1}.png")
            pix.save(output_path)
            image_paths.append(output_path)
        pdf_document.close()
    except Exception as e:
        print(f"  [ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
        return []
    return image_paths

def safe_cleanup_images(image_paths, max_retries=3):
    """Safely cleanup image files with retry logic."""
    for image_path in image_paths:
        if os.path.exists(image_path):
            for attempt in range(max_retries):
                try:
                    os.remove(image_path)
                    break
                except PermissionError:
                    print(f"  [WARNING] Could not delete {image_path} (attempt {attempt + 1}), retrying...")
                    time.sleep(1)  # Wait a bit before retrying
                    gc.collect()  # Force garbage collection
                except Exception as e:
                    print(f"  [ERROR] Failed to delete {image_path}: {e}")
                    break

def safe_cleanup_folder(folder_path, max_retries=3):
    """Safely cleanup temporary folder with retry logic."""
    if os.path.exists(folder_path):
        for attempt in range(max_retries):
            try:
                shutil.rmtree(folder_path)
                break
            except PermissionError:
                print(f"  [WARNING] Could not delete folder {folder_path} (attempt {attempt + 1}), retrying...")
                time.sleep(2)
                gc.collect()
            except Exception as e:
                print(f"  [ERROR] Failed to delete folder {folder_path}: {e}")
                break

def ocr_with_gemini_retry(image_paths, instruction, max_retries=3, base_delay=5):
    """Processes images with Gemini OCR with retry logic for timeout errors."""
    
    # Close PIL images properly to avoid file locks
    images = []
    try:
        for path in image_paths:
            img = Image.open(path)
            # Convert to bytes to avoid file locks
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='PNG')
            img_byte_arr.seek(0)
            images.append(Image.open(img_byte_arr))
            img.close()  # Close the original file handle
    except Exception as e:
        print(f"  [ERROR] Failed to load images: {e}")
        return None
    
    prompt = f"""
    {instruction}
    
    These are pages from a PDF document. Extract all text content while preserving the structure.
    Pay special attention to tables, columns, headers, and any structured content.
    Maintain paragraph breaks and formatting.
    """
    
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1} to process with Gemini...")
            response = model.generate_content([prompt, *images])
            return response.text
            
        except Exception as e:
            error_msg = str(e).lower()
            if "504" in error_msg or "deadline" in error_msg or "timeout" in error_msg:
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** attempt)  # Exponential backoff
                    print(f"  [WARNING] Timeout error (attempt {attempt + 1}). Retrying in {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for timeout error: {e}")
                    return None
            elif "quota" in error_msg or "rate" in error_msg:
                if attempt < max_retries - 1:
                    delay = 60  # Wait longer for quota issues
                    print(f"  [WARNING] Rate limit/quota error. Waiting {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for rate limit: {e}")
                    return None
            else:
                print(f"  [ERROR] Unexpected error: {e}")
                return None
    
    return None

def ocr_complex_document(image_paths):
    """Creates the specific prompt and calls the Gemini OCR function with retry."""
    instruction = """
        اقرأ النص القانوني أو حكم اللجنة المرفق بعناية تامة، واستخرج تفصيلاً كاملاً للقرار وفق الشكل التالي. التزم بالقواعد والتعليمات أدناه حرفياً.

        التزامات عامة:
        - استند فقط إلى النص الأصلي كما ورد في المستند؛ لا تضف أي تفسير، استنتاج، أو معلومات من خارج النص.
        - استخدم اللغة العربية الفصحى في كل الحقول.
        - لا تذكر أي مواد قانونية أو أرقام مواد/مراجع نظامية.
        - لا تذكر أي مبالغ مالية أو أرقام نقدية في أي حقل.
        - إذا كان أي حقل غير موجود صراحة في النص، اتركه كسلسلة فارغة "".
        - احتفظ بأسلوب قانوني ورسمي، لكن صِف بوضوح وبالتفصيل ما ورد في القرار عن وجهات النظر والأسباب.
        - إذا كان النص المستخرج غير واضح أو به غموض في المعنى، انسخه كما هو دون تعديل أو تفسير أو إعادة صياغة لتجنّب أي سوء فهم.
        - لا تضف أي نص خارج بنية JSON النهائية.

        هيكل الإخراج (أعد النتيجة بنفس البنية أدناه تمامًا):

        {
        "تفصيل_القرار": {
            "رقم_القرار_النهائي": "استخرج رقم القرار النهائي كما ورد نصاً. يكون موجود في رأس أول صفحة ويبدأ بـ IR أو VA.",
            "اسم_الدائرة_الابتدائية": "استخرج اسم دائرة الفصل الابتدائية كما ورد في النص.",
            "اسم_الدائرة_النهائية": "استخرج اسم الدائرة الاستئنافية أو اللجنة النهائية كما ورد في النص.",
            "اللجنة_الابتدائية": "سرد تفصيلي لقرار اللجنة الابتدائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
            "اللجنة_النهائية": "سرد تفصيلي لقرار اللجنة النهائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
            "البنود_محل_الدعوى": [
            {
                "اسم_البند": "انسخ اسم وعبارة البند كما وردت في القرار (عنوان البند).",
                "نبذة_مختصرة_عن_الاعتراض": "خلاصة وجيزة (جملة أو اثنتان) تشرح طبيعة الاعتراض على هذا البند كما وردت في القرار.",
                "وجهة_نظر_المكلف_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن دفوع المكلف/الشركة بهذا البند: الحجج، الوقائع والمستندات التي استند إليها، المطالبات والإشارات الزمنية أو العقدية أو المحاسبية أو الفنية التي ذكرها المكلف. انسخ النص قدر الإمكان مع إعادة ترتيب بسيط لقراءة منطقية إذا لزم، لكن لا تُحوّل المعنى ولا تبتّ من النص الأصلي.",
                "وجهة_نظر_الهيئة_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن موقف الهيئة بهذا البند: التبريرات، الأدلة أو النتائج الرقابية، طريقة تفسير الهيئة للوقائع والعقود والسجلات، وأي حجج فنية أو إجرائية ذكرتها الهيئة.",
                "رأي_اللجنة_الابتدائية_ومبرراته": "انقل نص قرار اللجنة الابتدائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. اذكر تفاصيل القرار ونتيجته كما وردت نصاً.وفي حاله عدم توفرها بشكل صريح قم باستنتاجها بناء علي المعطيات",
                "رأي_اللجنة_النهائية_ومبرراته": "انقل نص قرار اللجنة النهائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. لا تختصر، وأعد الأسباب كاملة أو بنحو قريب جداً مع الحفاظ على المعنى والتفصيل."
            }
            ],
            "خلاصة_نهائية": "جملة أو فقرة واحدة تُلخّص نتيجة القرار الكلية كما وردت (مثل: قبول الاستئناف شكلاً وإعادة الدعوى إلى دائرة الفصل للنظر في بنود محددة)، لكن بدون ذكر مواد قانونية أو مبالغ."
        }
        }

        قواعد تنفيذية إضافية:
        1. لا تُدرج في أي حقل عناوين مواد قانونية أو أرقام مواد أو نصوص نظامية.
        2. في حالة عدم ذكر قرار اللجنة الابتدائية قم باستنتاجه بناء على السياق.
        3. لا تذكر مبالغ مالية، ولا قيّم الخسائر أو التزامات نقدية.
        4. عند نقل وجهات النظر والأسباب، أذكر بوضوح ما استندت إليه كل جهة (مثلاً: "استندت الشركة إلى القوائم المالية المدققة والعقود المبرمة" أو "استندت الهيئة إلى نتائج الفحص المستندي") — لكن هذا فقط إذا ورد صراحة في النص.
        5. حافظ على الترتيب كما في القالب؛ لا تضف حقولًا جديدة ولا تزيل الحقول المطلوبة.
        6. أعد الناتج بصيغة JSON صالحة فقط ولا تضف أي تعليقات خارجها.
        7. قم بسرد تفصيل لقرار (اللجنة الابتدائية) في البداية وأذكر اسم اللجنة من واقع القرار.
        8. قم بسرد تفصيل لقرار (اللجنة النهائية) في البداية وأذكر اسم اللجنة من واقع القرار.
        9. لكل بند من البنود محل الدعوى، يجب أن يحتوي على (رأي ابتدائي) و(رأي نهائي) مستقلين مع المبررات كما وردت في النص.
        10. إذا كان النص المستخرج غير واضح أو به التباس، انسخه كما هو دون تعديل أو تفسير لتجنّب أي سوء فهم.
        """

    return ocr_with_gemini_retry(image_paths, instruction)

def clean_gemini_output(raw_text):
    """Cleans the raw text from Gemini to extract the JSON part."""
    if not raw_text:
        return "{}"
    
    # Find the start and end of the JSON block
    start_index = raw_text.find('{')
    end_index = raw_text.rfind('}')
    
    if start_index != -1 and end_index != -1:
        json_str = raw_text[start_index:end_index+1]
        return json_str
    
    # Return empty JSON if no valid block is found
    return "{}"

def save_progress(results, output_file):
    """Save current progress to avoid losing data."""
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"  ✅ Progress saved to {output_file}")
    except Exception as e:
        print(f"  [ERROR] Could not save progress: {e}")

def main():
    """
    Main function to orchestrate the PDF processing workflow.
    Saves all extracted data into a JSON file with progress saving.
    """
    # --- CONFIGURATION ---
    pdf_input_folder = r"C:\Users\AReda\Downloads\Rett\Decesions\done"
    temp_image_folder = r"C:\Users\AReda\Downloads\Rett\Decesions\temp_images"
    output_json_file = r"C:\Users\AReda\Downloads\Rett\Decesions\decisions.json"
    progress_file = r"C:\Users\AReda\Downloads\Rett\Decesions\progress_backup.json"
    # --- END CONFIGURATION ---

    results = []  # list of extracted dicts
    processed_files = set()  # Track processed files

    # Load existing progress if available
    if os.path.exists(progress_file):
        try:
            with open(progress_file, "r", encoding="utf-8") as f:
                results = json.load(f)
                processed_files = {item.get('Source_Filename', '') for item in results}
            print(f"Loaded {len(results)} previously processed files from progress backup.")
        except Exception as e:
            print(f"Could not load progress file: {e}")

    # Get PDF files
    pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
    remaining_files = [f for f in pdf_files if f not in processed_files]
    
    print(f"\nFound {len(pdf_files)} total PDF files.")
    print(f"Already processed: {len(processed_files)}")
    print(f"Remaining to process: {len(remaining_files)}")

    for i, filename in enumerate(remaining_files, 1):
        print(f"\n--- Processing '{filename}' ({i}/{len(remaining_files)}) ---")
        pdf_path = os.path.join(pdf_input_folder, filename)
        image_paths = []

        try:
            # 1. Convert PDF to images
            print("  Step 1: Converting PDF to images...")
            image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)

            if not image_paths:
                print(f"  [ERROR] No images were created for '{filename}'. Skipping.")
                continue

            # 2. Extract data with Gemini (with retry logic)
            print("  Step 2: Extracting text with Gemini AI...")
            raw_gemini_text = ocr_complex_document(image_paths)

            if raw_gemini_text is None:
                print(f"  [ERROR] Failed to extract text for '{filename}'. Skipping.")
                continue

            # 3. Parse JSON output
            print("  Step 3: Parsing AI output...")
            json_text = clean_gemini_output(raw_gemini_text)
            
            try:
                extracted_data = json.loads(json_text)
            except json.JSONDecodeError as e:
                print(f"  [ERROR] Invalid JSON output for '{filename}': {e}")
                print(f"  Raw output: {json_text[:200]}...")
                continue

            # Add source filename
            extracted_data['Source_Filename'] = filename

            # 4. Append to results list
            results.append(extracted_data)

            print(f"  ✅ Successfully processed '{filename}'")
            
            # Save progress every 5 files
            if len(results) % 5 == 0:
                save_progress(results, progress_file)

        except Exception as e:
            print(f"  [!!!] Unexpected error while processing '{filename}': {e}")
            continue

        finally:
            # Cleanup temp images more safely
            if image_paths:
                print("  Cleaning up temporary files...")
                safe_cleanup_images(image_paths)
                time.sleep(1)  # Give system time to release file handles
            
            # Clean up temp folder if it exists
            safe_cleanup_folder(temp_image_folder)

        # Add a small delay between files to avoid rate limiting
        if i < len(remaining_files):
            print("  Waiting before next file...")
            time.sleep(2)

    print("\n--- All PDF files have been processed. ---")

    # Save final results
    try:
        with open(output_json_file, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✅ Saved extracted data to {output_json_file}")
        
        # Clean up progress file
        if os.path.exists(progress_file):
            os.remove(progress_file)
            
    except Exception as e:
        print(f"[ERROR] Could not save final JSON file: {e}")

    return results


# Example usage
if __name__ == "__main__":
    extracted_chunks = main()
    print(f"\nProcessing complete! Total extracted documents: {len(extracted_chunks)}")
    if extracted_chunks:
        print("\nSample Extracted Data (first item):")
        print(json.dumps(extracted_chunks[0], ensure_ascii=False, indent=2))

c:\Users\AReda\anaconda3\envs\finbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Found 250 total PDF files.
Already processed: 0
Remaining to process: 250

--- Processing '238397-2024-R - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf' (1/250) ---
  Step 1: Converting PDF to images...
  Step 2: Extracting text with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed '238397-2024-R - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf'
  Cleaning up temporary files...
  Waiting before next file...

--- Processing 'R-167901-2023 - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf' (2/250) ---
  Step 1: Converting PDF to images...
  Step 2: Extracting text with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed 'R-167901-2023 - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضا